# 06 — تحلیل رویداد و هم‌ترازی مالی

این Notebook فازهای سیزدهم و چهاردهم checklist.md (§25، §26 و §27) را روایت می‌کند.  
**بخش ۱:** Event Analysis — فراخوانی خروجی‌های `src/event_analysis/event_study.py`  
**بخش ۲:** Financial Alignment — ارجاع به `notebooks/financial/02_financial_social_alignment.ipynb`

**ترتیب اجرا:** پس از `05_descriptive_and_temporal_analysis.ipynb`

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=None):
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "event_analysis").exists() or (candidate / "config" / "config.yaml").exists():
            return candidate
    raise FileNotFoundError("Project root not found — run from within the media-sentiment-pipeline repo.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"ROOT = {ROOT}")

In [ ]:
import pandas as pd

REAL_INPUT   = ROOT / "data" / "processed" / "annotated_dataset.parquet"
SAMPLE_INPUT = ROOT / "data" / "processed" / "annotated_dataset.sample.parquet"
EVENT_DIR    = ROOT / "outputs" / "tables" / "event_analysis"

real_data_ready = REAL_INPUT.exists() and REAL_INPUT.stat().st_size > 0
sample_ready    = SAMPLE_INPUT.exists() and SAMPLE_INPUT.stat().st_size > 0

print({
    "real_annotated_dataset_exists": real_data_ready,
    "sample_fallback_exists": sample_ready,
    "event_analysis_dir_exists": EVENT_DIR.exists(),
})

---
## بخش ۱ — Event Analysis (§25)

### قرارداد آماری — §25

- **رویدادها پیش‌ثبت‌شده‌اند:** هر ۴ رویداد در `docs/event_registry_v3.md §4` قبل از اجرای تحلیل ثبت شده‌اند — cherry-picking ممنوع (§45 checklist).
- **EV-001** (`study_anchor`): آغاز جنگ — توصیفی‌فقط، بدون پنجره قبل/بعد.
- **EV-016، EV-025، EV-031** (`primary_confirmatory`): آزمون before/after سهم stance.
- **واحد تحلیل:** سهم `stance_label` روی ردیف‌های `annotation_status=="ok"` که `target==target_id` رویداد باشند.
- **آماره اصلی:** `share_diff = p_after - p_before` با **95% CI** برای تفاضل دو نسبت مستقل (فرمول بسته — بدون scipy).
- **تحلیل جداگانه:** X، Reddit و YouTube **جداگانه** — هیچ pooling پلتفرمی در نتیجه اصلی نیست.
- ⚠️ **تمام نتایج «همراهی زمانی» هستند، نه اثر علّی** (قانون §29 checklist — ستون‌ها عمداً `share_diff` نام دارند نه `effect` یا `impact`).
- **Placebo:** همان آزمون در تاریخی که هیچ رویدادی ثبت نشده — اگر نتیجه Placebo مشابه نتیجه اصلی باشد، یافته اصلی باید با احتیاط بیشتری تفسیر شود.
- **حساسیت‌ها:** پنجره زمانی باریک‌تر، حذف near-duplicate، حذف بزرگ‌ترین source.

In [ ]:
EVENT_FILES = {
    "main":                     EVENT_DIR / "event_study_main.csv",
    "composition":              EVENT_DIR / "event_study_composition.csv",
    "placebo":                  EVENT_DIR / "event_study_placebo.csv",
    "sensitivity_window":       EVENT_DIR / "event_study_sensitivity_window.csv",
    "sensitivity_robustness":   EVENT_DIR / "event_study_sensitivity_robustness.csv",
    "anchor_descriptive":       EVENT_DIR / "event_study_anchor_descriptive.csv",
}

event_outputs_exist = all(p.exists() for p in EVENT_FILES.values())
print(f"خروجی‌های event_study موجود: {event_outputs_exist}")
for label, path in EVENT_FILES.items():
    print(f"  {label}: {'✅' if path.exists() else '❌'} {path.name}")

In [ ]:
if event_outputs_exist:
    ev_main     = pd.read_csv(EVENT_FILES["main"])
    ev_comp     = pd.read_csv(EVENT_FILES["composition"])
    ev_placebo  = pd.read_csv(EVENT_FILES["placebo"])
    ev_sens_win = pd.read_csv(EVENT_FILES["sensitivity_window"])
    ev_sens_rob = pd.read_csv(EVENT_FILES["sensitivity_robustness"])
    ev_anchor   = pd.read_csv(EVENT_FILES["anchor_descriptive"])
    print("STATUS: loaded_from_precomputed_csvs")
elif real_data_ready or sample_ready:
    from src.event_analysis.event_study import build_all
    from src.temporal_analysis.common import load_annotated_dataset

    input_path = REAL_INPUT if real_data_ready else SAMPLE_INPUT
    print(f"در حال محاسبه از: {input_path.name} ...")

    df = load_annotated_dataset(input_path)
    tables = build_all(df)

    EVENT_DIR.mkdir(parents=True, exist_ok=True)
    for name, tbl in tables.items():
        tbl.to_csv(EVENT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")

    ev_main     = tables["event_study_main"]
    ev_comp     = tables["event_study_composition"]
    ev_placebo  = tables["event_study_placebo"]
    ev_sens_win = tables["event_study_sensitivity_window"]
    ev_sens_rob = tables["event_study_sensitivity_robustness"]
    ev_anchor   = tables["event_study_anchor_descriptive"]
    print(f"STATUS: computed — {len(df):,} ردیف")
else:
    print("STATUS: pending_annotated_dataset")
    print(f"ورودی مورد انتظار: {REAL_INPUT}")
    ev_main = ev_comp = ev_placebo = ev_sens_win = ev_sens_rob = ev_anchor = None

### ۱.۱ — EV-001: آغاز حملات (study_anchor) — توصیف پسارویداد

EV-001 پایه‌ی زمانی پروژه است (هفته W01). چون هیچ پنجره «قبل» در بازه پروژه وجود ندارد، هیچ آزمون before/after روی آن اجرا نمی‌شود — فقط حجم و ترکیب برچسب در ۱۴ روز پس از آن گزارش می‌شود.

In [ ]:
if ev_anchor is not None:
    display(ev_anchor)
else:
    print("STATUS: pending_annotated_dataset")

### ۱.۲ — نتایج اصلی: سهم Stance قبل و بعد از رویداد (به‌تفکیک پلتفرم)

ستون‌های کلیدی:  
- `share_diff`: تفاضل سهم بعد منهای قبل — مثبت یعنی افزایش، منفی یعنی کاهش  
- `ci_low` / `ci_high`: فاصله اطمینان ۹۵٪ برای این تفاضل  
- `p_before` / `p_after`: سهم stance مرتبط در هر دوره  
- `n_before_target` / `n_after_target`: حجم ردیف‌های دارای target در هر دوره

⚠️ **تمام اعداد زیر «همراهی زمانی» هستند، نه اثر علّی.**

In [ ]:
if ev_main is not None:
    cols_display = [
        "event_id", "title_fa", "platform", "window_days",
        "n_before_target", "n_after_target",
        "p_before", "p_after", "share_diff", "ci_low", "ci_high",
        "expected_direction_fa",
    ]
    available = [c for c in cols_display if c in ev_main.columns]
    for event_id, grp in ev_main.groupby("event_id"):
        print(f"\n=== {event_id} ===")
        display(grp[available].reset_index(drop=True))
else:
    print("STATUS: pending_annotated_dataset")

### ۱.۳ — بررسی Composition در پنجره رویداد

یک تغییر در سهم Stance ممکن است نتیجه تغییر نگرش باشد — یا نتیجه تغییر ترکیب محتوا (مثلاً پلتفرم‌های مختلف قبل و بعد). این جدول امکان تشخیص را فراهم می‌کند.

In [ ]:
if ev_comp is not None:
    for event_id, grp in ev_comp.groupby("event_id"):
        print(f"\n=== {event_id} ===")
        display(grp[["window_label", "platform", "n_records", "platform_share_of_window", "total_records_in_window"]]
                .sort_values(["window_label", "platform"])
                .reset_index(drop=True))
else:
    print("STATUS: pending_annotated_dataset")

### ۱.۴ — Placebo: آزمون در تاریخ ثبت‌نشده

اگر همان آزمون before/after در تاریخی که **هیچ رویدادی ثبت نشده** نتیجه مشابهی نشان دهد، این نشانه‌ای است که نتیجه اصلی به تغییر تصادفی یا تأثیر عوامل دیگر بازمی‌گردد — نه به رویداد ثبت‌شده.  

⚠️ Placebo یک رویداد واقعی نیست و هرگز به‌عنوان آن گزارش نمی‌شود.

In [ ]:
if ev_placebo is not None:
    cols_display = [
        "event_id", "platform", "window_days",
        "n_before_target", "n_after_target",
        "p_before", "p_after", "share_diff", "ci_low", "ci_high",
    ]
    available = [c for c in cols_display if c in ev_placebo.columns]
    display(ev_placebo[available].reset_index(drop=True))
else:
    print("STATUS: pending_annotated_dataset")

### ۱.۵ — حساسیت‌ها

**§25 checklist:** حداقل سه بررسی حساسیت:
1. پنجره زمانی باریک‌تر (`sensitivity_window`)
2. حذف near-duplicate (`sensitivity_excl_near_dup`)
3. حذف بزرگ‌ترین source (`sensitivity_excl_top_source`)

In [ ]:
print("--- حساسیت: پنجره زمانی باریک‌تر ---")
if ev_sens_win is not None:
    cols_display = ["event_id", "platform", "window_label", "window_days",
                    "n_before_target", "n_after_target", "share_diff", "ci_low", "ci_high"]
    available = [c for c in cols_display if c in ev_sens_win.columns]
    display(ev_sens_win[available].reset_index(drop=True))
else:
    print("STATUS: pending_annotated_dataset")

In [ ]:
print("--- حساسیت: حذف near-duplicate و بزرگ‌ترین source ---")
if ev_sens_rob is not None:
    cols_display = ["event_id", "platform", "window_label", "window_days",
                    "n_before_target", "n_after_target", "share_diff", "ci_low", "ci_high"]
    available = [c for c in cols_display if c in ev_sens_rob.columns]
    display(ev_sens_rob[available].reset_index(drop=True))
else:
    print("STATUS: pending_annotated_dataset")

### یادداشت تفسیری §25 / §29

- **«همراهی زمانی»** — نه اثر علّی: افزایش یا کاهش سهم Stance همزمان با یک رویداد به معنای رابطه علّی نیست.
- فاصله اطمینان از صفر گذر می‌کند (یعنی CI شامل صفر است)؟ → نتیجه با داده موجود قابل تمایز از تصادف نیست.
- اگر نتیجه Placebo (§۱.۴) از نظر اندازه مشابه نتیجه اصلی باشد، نتیجه اصلی باید با احتیاط بیشتری ارائه شود.
- حداکثر ۲۱ هفته داده: توان آماری محدود است — نبود معناداری اثبات نبود رابطه نیست.

---
## بخش ۲ — هم‌ترازی مالی (§26 و §27)

تحلیل کامل در `notebooks/financial/02_financial_social_alignment.ipynb` پیاده شده.  
این بخش خلاصه‌ای از نتایج آن را بارگذاری و ارائه می‌دهد.

### قرارداد آماری — §26 و §27

- **داده مالی:** تغییر هفتگی (نه سطح قیمت) برای جلوگیری از Spurious correlation بین سری‌های غیر‌ثابت.
- **روش اصلی:** Spearman correlation با Lag صفر، یک، و دو هفته.
- **Outcome اجتماعی هفته `t`** با **تغییر مالی هفته `t + lag`** مقایسه می‌شود.
- **فاصله اطمینان:** Bootstrap درصدی بر پایه جفت‌های هفته.
- **p-value:** ۹٬۹۹۹ Permutation جفت‌شده.
- **اصلاح چندگانگی:** Benjamini–Hochberg FDR.
- **حداقل داده:** ۱۰ هفته جفت‌شده.
- **تفسیر:** Association زمانی — نه رابطه علّی.

In [ ]:
FINANCIAL_NOTEBOOK = ROOT / "notebooks" / "financial" / "02_financial_social_alignment.ipynb"
FINANCIAL_RESULT   = ROOT / "outputs" / "tables" / "financial" / "financial_social_correlation_results_v1.csv"

print({
    "financial_notebook_exists":       FINANCIAL_NOTEBOOK.exists(),
    "financial_result_csv_exists":      FINANCIAL_RESULT.exists(),
    "financial_result_csv_size_bytes":  FINANCIAL_RESULT.stat().st_size if FINANCIAL_RESULT.exists() else 0,
})

In [ ]:
fin_ready = FINANCIAL_RESULT.exists() and FINANCIAL_RESULT.stat().st_size > 0

if fin_ready:
    financial_results = pd.read_csv(FINANCIAL_RESULT)
    print(f"STATUS: loaded — {len(financial_results):,} آزمون ثبت‌شده")
else:
    print("STATUS: pending_social_outcomes")
    print(f"نوت‌بوک مالی: {FINANCIAL_NOTEBOOK}")
    print("برای تولید خروجی: نوت‌بوک financial/02 را پس از آماده‌شدن social_weekly_outcomes_v1.csv اجرا کنید.")
    financial_results = None

### ۲.۱ — خلاصه آزمون‌های اصلی Spearman (Lag 0، 1، 2)

In [ ]:
if financial_results is not None:
    primary = financial_results[
        (financial_results["analysis_variant"] == "primary_all_weeks") &
        (financial_results["method"] == "spearman")
    ]
    display_cols = [
        "platform", "outcome_id", "target_id", "asset_id",
        "financial_lag_weeks", "n_paired_weeks",
        "coefficient", "ci_low", "ci_high",
        "p_value_raw", "p_value_bh_fdr", "status",
    ]
    available = [c for c in display_cols if c in primary.columns]
    display(primary[available].sort_values(["platform", "financial_lag_weeks"]).reset_index(drop=True))
else:
    print("STATUS: pending_social_outcomes")

### ۲.۲ — حساسیت‌های مالی

In [ ]:
if financial_results is not None:
    sens = financial_results[
        financial_results["analysis_variant"].str.startswith("sensitivity")
    ]
    display_cols = [
        "platform", "analysis_variant", "method", "outcome_id", "target_id",
        "asset_id", "financial_lag_weeks", "n_paired_weeks",
        "coefficient", "ci_low", "ci_high", "p_value_raw", "p_value_bh_fdr", "status",
    ]
    available = [c for c in display_cols if c in sens.columns]
    display(sens[available].sort_values(["analysis_variant", "platform", "financial_lag_weeks"]).reset_index(drop=True))
else:
    print("STATUS: pending_social_outcomes")

### یادداشت تفسیری §27

- ضریب Spearman و فاصله اطمینان از p-value مهم‌ترند — اندازه رابطه و دقت تخمین را نشان می‌دهند.
- نتیجه با `p_value_bh_fdr < 0.05` فقط به‌عنوان شواهد **ارتباط زمانی در نمونه مشاهده‌شده** گزارش می‌شود.
- نبود نتیجه معنادار اثبات نبود رابطه نیست — به‌ویژه با حداکثر ۲۱ هفته جفت‌شده.
- تحلیل کامل، روایت، و نمودارها در `notebooks/financial/02_financial_social_alignment.ipynb` موجودند.

**ارجاع مستقیم:**

In [ ]:
print(f"نوت‌بوک مالی کامل:\n  {FINANCIAL_NOTEBOOK}")
print(f"\nفایل نتایج:\n  {FINANCIAL_RESULT}")
print(f"\nنوت‌بوک موجود: {'✅' if FINANCIAL_NOTEBOOK.exists() else '❌'}")

---
## خلاصه اجرا

| بخش | وضعیت | منبع کد |
|-----|--------|----------|
| §25 Event Analysis | ✅ آماده (شرطی) | `src/event_analysis/event_study.py` + `event_registry.py` |
| §26 هم‌ترازی مالی | ✅ آماده در نوت‌بوک مالی | `notebooks/financial/02_financial_social_alignment.ipynb` |
| §27 آزمون مالی | ✅ آماده (شرطی) | `notebooks/financial/02_financial_social_alignment.ipynb` |

**اجرای کامل و نهایی:** پس از رسیدن `annotated_dataset.parquet` واقعی از Pipeline A و تولید `social_weekly_outcomes_v1.csv`.